# AI - Theory To Practice CA1

### 1.1 Dateset Requirements:
1. 1,000 observations
2. 10 input variables before cleaning
3. Both Numerical & Categorical variables
4. Publicly accessible source - https://www.basketball-reference.com/players/c/curryst01/gamelog/2010

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
steph = pd.read_excel("Steph_curry.xlsx")
print("Data Shape:", steph.shape)

### 1.2 Problem Definition
- The objective of this project for our models to predict if Stephen Curry can score more than 24.8 points against certain teams, based on his career against each team. 
- This is a Classification Problem, We want a percentage probability result.
- The Target Variable is PTS. (PTS)
- The Predictor Variables are Opp (Opponent), FG%(Field Goal Percentage), eFG%(Effective Field Goal Percentage), TOV(Turnover).
- This project is useful for Sports fans, Gambling Sites, Valuable Insights and potential Investment opportunites.

In [ ]:
features = steph[["Date", "Opp", "FG%", "eFG%","FT", "TOV","GmSc", "+/-"]]

In [ ]:
steph.head()

### 2 Data Understanding & Exploratory Data Analysis (EDA)

-  Dataset Size: 1,444 Rows & 36 Columns 

In [ ]:
print("Data Shape", steph.shape)

- Variable Types: 27 Float, 5 String Columns, 1 Datetime & 1 Object

In [ ]:
steph.dtypes.value_counts()

- Statistical Summary: Quartiles, Mean, Standard Deviation, Min, Max.

In [ ]:
features.describe()

- Cardinality (Unique Values)

In [ ]:
print(features.nunique())

- IQR (Inner Quartile Range)

In [ ]:
features.select_dtypes(include="number").quantile(0.75) - features.select_dtypes(include="number").quantile(0.25)

- Missing Values Summary

In [ ]:
missing_features_count = features.isnull().sum()
missing_features_percentage = features.isnull().mean() * 100

missing_features_summary = pd.DataFrame({
    "Missing Count": missing_features_count,
    "Missing Percentage": missing_features_percentage
})

print(missing_features_summary)

### Average 3 Pointers Made VS Each Team
- Show's us The average amount of 3 pointers scored by Stephen Curry against each team.
- Important as 3 Pointers are what Steph is most known for.
- Potentially some insight into how he will perform based on team.

In [ ]:
average_fg = steph.groupby("Opp")["3P"].mean()

plt.figure(figsize=(10, 8))
plt.margins(y = 0.08), plt.grid(axis="x")
plt.barh(average_fg.index, average_fg.values)
plt.xlabel("Average Made 3 Pointers Per Game")
plt.ylabel("Opponent")
plt.title("Average 3 Pointers Made VS Each Opponent")

plt.show()

### Average 2 Pointers Made VS Each Team
- Show's us The average amount of 2 pointers scored by Stephen Curry against each team.
- Important as 2 pointers are most common in the sport of Basketball.
- Potentially some insight into how he will perform based on team.

In [ ]:
average_fg = steph.groupby("Opp")["2P"].mean()

plt.figure(figsize=(10, 8))
plt.margins(y = 0.08), plt.grid(axis="x")
plt.barh(average_fg.index, average_fg.values)
plt.xlabel("Average Made 3 Pointers Per Game")
plt.ylabel("Opponent")
plt.title("Average 2 Pointers Made VS Each Opponent")

plt.show()

### Average Free Throws Made VS Each Team
- Show's us The average amount of Free Throws scored by Stephen Curry against each team.
- Important as Free Throws are the only other way of scoring.
- Potentially some insight into how he will perform based on team.

In [ ]:
average_fg = steph.groupby("Opp")["FT"].mean()

plt.figure(figsize=(10, 8))
plt.margins(y = 0.08), plt.grid(axis="x")
plt.barh(average_fg.index, average_fg.values)
plt.xlabel("Average Made 3 Pointers Per Game")
plt.ylabel("Opponent")
plt.title("Average Free Throws Made VS Each Opponent")

plt.show()

### Distribution of 3 Pointers Made in Career
- Show's us the distribution of 3 pointers made, throughout Steph's career.
- Important as it shows the distribution of where the 3 pointers are.
- Potentially some insight into how he will perform based performance.

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(steph["3P"], bins=range(
    int(steph["3P"].min()),
    int(steph["3P"].max()) + 2
), edgecolor="black")
plt.xlabel("3-Pointers Made")
plt.ylabel("Number of Games")
plt.title("Distribution of Steph's 3-Pointers")
plt.show()

### Top Field Goal Games
- Show's us The best performances in Steph's Career.
- Important as some of these may be considered outliers.
- Potentially some insight outliers in dataset.

In [ ]:
steph.sort_values(by="FG" , ascending=False).head(10).plot(
    kind='scatter', 
    x="FG", 
    y="Opp", 
    figsize=(10, 6),
    grid=True,
    s=100,
    color="darkblue")
plt.title("Top Field Goal Games")
plt.xlabel("Baskets made")
plt.ylabel("Opponent")
plt.gca().invert_yaxis()
plt.show()


## 3.0 Data Cleaning & Preperation

### 3.1 General data-quality checks
Duplicates - No duplicate rows in dataset.

In [ ]:
duplicates = steph[steph.duplicated(keep=False)]

Data Type - Changing from string to date

In [ ]:
steph['Date'] = pd.to_datetime(steph['Date'])
print(steph['Date'].head(5))

Impossible, Invalid or Inconsistent Values:

- 3 pointers made MUST be under 3 pointers attempted
- 2 pointers made MUST be under 2 pointers attempted
- Free Throw's made MUST be under Free Throw's attempted

In [ ]:
Invalid = steph[steph['3P'] > steph['3PA']]
print("Invalid Three-Point Games: ", Invalid)

Invalid = steph[steph['2P'] > steph['2PA']]
print("Invalid Three-Point Games: ", Invalid)

Invalid = steph[steph['FT'] > steph['FTA']]
print("Invalid Three-Point Games: ", Invalid)

High-Cardinality - Date is our highest Cardinality, TOV is our lowest.

In [ ]:
print(features.nunique())

### 3.2 Missing Data

Missing Values - Many missing values which need investigating.

In [ ]:
print(features.isnull().sum())

Date + Opp - Seem's to be blank rows within the dataset. 

In [ ]:
display(features[features['Opp'].isna()].head(5))

- Removing all empty rows, as they have no team for the target variable

In [ ]:
features = features.dropna(subset=["Opp", "Date"])

FG, eFG% - In Excel sheet although missing in translation. 

In [ ]:
display(features[features['FG%'].isna()].head(5))

In [ ]:
display(features[features['eFG%'].isna()].head(5))

Numerical - Making each figure the mean.
- Simple to implement
- Preserves Data size
- Appropriate as Steph played the games, he wasnt injured.

In [ ]:
cols = ['FG%', 'eFG%', 'FT', 'TOV', 'GmSc', '+/-']

features[cols] = features[cols].fillna(
    features[cols].mean()
)

### 3.3 Outliers

